##### Copyright 2025 Google LLC.

In [1]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemini API: Similarity Search using Couchbase

<a target="_blank" href="https://colab.research.google.com/github/google-gemini/cookbook/blob/main/examples/couchbase/couchbase_similarity_search.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" height=30/></a>

## Overview

The [Gemini API](https://ai.google.dev/models/gemini) provides access to a family of generative AI models for generating content and solving problems. These models are designed and trained to handle both text and images as input.

[Couchbase](https://www.couchbase.com/) is an award-winning distributed NoSQL developer database platform that delivers unmatched versatility, performance, scalability, and financial value for your critical applications. Couchbase embraces AI with coding assistance for developers, plus AI services for building applications that include RAG-powered agents, real-time analytics, and cloud-to-edge vector search.

In this notebook, you'll learn how to perform a similarity search on data from a website with the help of Gemini API and Couchbase.


# Before you start

## Create and Deploy Your Free Tier Operational cluster on Capella

To get started with Couchbase Capella, create an account and use it to deploy a forever free tier operational cluster. This account provides you with a environment where you can explore and learn about Capella with no time constraint.

To know more, please follow the [instructions](https://docs.couchbase.com/cloud/get-started/create-account.html).

### Couchbase Capella Configuration

When running Couchbase using [Capella](https://cloud.couchbase.com/sign-in), the following prerequisites need to be met.

* Create the [database credentials](https://docs.couchbase.com/cloud/clusters/manage-database-users.html) to access the travel-sample bucket (Read and Write) used in the application.
* [Allow access](https://docs.couchbase.com/cloud/clusters/allow-ip-address.html) to the Cluster from the IP on which the application is running.

## Installation

Install Google's python client SDK for the Gemini API, `google-genai`. Next, install Couchbase's Python client SDK, `couchbase`.

In [ ]:
!pip install -q couchbase==4.3.5 google-genai==1.11.0 beautifulsoup4==4.12.2

## Configure your API key

To run the following cell, your API key must be stored it in a Colab Secret named `GOOGLE_API_KEY`. If you don't already have an API key, or you're not sure how to create a Colab Secret, see [Authentication](https://github.com/google-gemini/cookbook/blob/main/quickstarts/Authentication.ipynb) for an example.


In [4]:
from google import genai

# from google.colab import userdata
# GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
# client = genai.Client(api_key=GOOGLE_API_KEY)

# If you are not running on Colab, you can set the API key manually
client = genai.Client(api_key="your-api-key")

# Imports

In [5]:
from pathlib import Path
from datetime import timedelta

from couchbase.auth import PasswordAuthenticator
from couchbase.cluster import Cluster
from couchbase.options import ClusterOptions
import couchbase.search as search
from couchbase.options import SearchOptions
from couchbase.vector_search import VectorQuery, VectorSearch
import uuid

from bs4 import BeautifulSoup
from urllib.request import urlopen

from pprint import pprint

## Basic steps

In this tutorial, you'll implement the three main components of vector search:

1. Setting Up Couchbase
    
    This step involves setting up configurations and credentials for Couchbase, and connecting to Couchbase. Finally we create a vector search index on Couchbase .

2. Generate and Store Embeddings in Couchbase

    In this step, we generate embeddings from a sustainability report using the embedding models available on the `genai` package. These documents are then stored in Couchbase.

3. Query the Index

    Query the index using a query string to return the top `n` neighbors of the query.

You'll learn more about these stages in the upcoming sections while implementing the application.

# 1. Setting Up Couchbase

### Configuring Couchbase

To use Couchbase, you must configure the host, username, password of your cluster, along with the bucket, scope, collection and index name. It is recommended to set the scope and collection name as follows:

* scope: shared
* collection: gemini

This ensures that you can easily import the index definition in Couchbase without any modification, which we will cover in a later section. Make sure that the bucket, scope and collection are already created on Couchbase, for which you can follow [these docs](https://docs.couchbase.com/cloud/clusters/data-service/about-buckets-scopes-collections.html).

In [6]:
import getpass

couchbase_cluster_url = input("Cluster URL:")
couchbase_username = input("Couchbase username:")
couchbase_password = getpass.getpass("Couchbase password:")
couchbase_bucket = input("Couchbase bucket:")
couchbase_scope = input("Couchbase scope:")
couchbase_collection = input("Couchbase collection:")

### Connecting to Couchbase

In [7]:
auth = PasswordAuthenticator(
    couchbase_username,
    couchbase_password
)

cluster = Cluster(couchbase_cluster_url, ClusterOptions(auth))
cluster.wait_until_ready(timedelta(seconds=30))

bucket = cluster.bucket(couchbase_bucket)
scope = bucket.scope(couchbase_scope)
collection = scope.collection(couchbase_collection)

### Creating Couchbase Vector Search Index
In order to store Gemini embeddings onto a Couchbase Cluster, a vector search index needs to be created first. We included a sample index definition that will work with this tutorial in the `gemini_index.json` file. The definition can be used to create a vector index on Couchbase Capella, on more information on vector indexes, please read [Create a Vector Search Index on Capella](https://docs.couchbase.com/cloud/vector-search/create-vector-search-index-ui.html). Particularly, you can check [this section](https://docs.couchbase.com/cloud/current/vector-search/create-vector-search-index-ui.html#example) on how you can import `gemini_index.json`.

Once you create the index, execute the following cell.

In [8]:
search_index_name = couchbase_bucket + ".shared.gemini_index"
search_index = cluster.search_indexes().get_index(search_index_name)

# 2. Generate and Store Embeddings in Couchbase

### Initialize the embedding model

To create the embeddings from the website data, you'll use the **embedding-001** model, which supports creating embeddings from text.

To use the embedding model, you have to use the `embed_content` function from the `google-generativeai` package. To learn more about the embedding model, read the [model documentation](https://ai.google.dev/gemini-api/docs/models/gemini#embedding).

One of the arguments passed to the embedding function is `task_type`. Speciefying the `task_type` parameter ensures the model produces appropriate embeddingsfor the expected task and inputs. It is a string that can take on one of the following values:

| task_type	  |  Description |
|---|---|
| `RETRIEVAL_QUERY` | Specifies the given text is a query in a search or retrieval setting. |
| `RETRIEVAL_DOCUMENT` | Specifies the given text is a document in a search or retrieval setting. |  
| `SEMANTIC_SIMILARITY` | Specifies the given text will be used for Semantic Textual Similarity (STS). |  
| `CLASSIFICATION` | Specifies that the embeddings will be used for classification. |
| `CLUSTERING` | Specifies that the embeddings will be used for clustering. |

In [9]:
from google.genai import types

embedding_model = "models/embedding-001"

# Function to convert text to embeddings
def make_embed_text_fn(text, model=embedding_model,
                       task_type="retrieval_document"):
    embedding = client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(
          task_type=task_type,
        )
    )
    return embedding.embeddings

### Embedding Documents

We can use Gemini to generate embeddings and store it in Couchbase for retrieval. In the cell below, we read an AI sustainability report, preprocess it, and generate embeddings.


In [10]:
url = "https://blog.google/outreach-initiatives/sustainability/"\
      "report-ai-sustainability-google-cop28/"
html = urlopen(url).read()
soup = BeautifulSoup(html, features="html.parser")

# Remove all script and style elements
for script in soup(["script", "style"]):
    script.extract()    # Self-destruct

# Get the text
text_content = soup.get_text()

# The text content between the substrings "Later this month at COP28" to
# "POSTED IN:" is relevant for this tutorial. You can use Python's `split()`
# to select the required content.
text_content_1 = text_content.split("Later this month at COP28",1)[1]
final_text = text_content_1.split("POSTED IN:",1)[0]

texts = final_text.split(".")

documents = []

# Convert text into a chunk of 3 sentences.
for i in range(0, len(texts), 3):
    documents.append(" ".join(texts[i:i+3]))

embeddings = make_embed_text_fn(documents)

print("Number of embeddings: " + str(len(embeddings)))

Number of embeddings: 14


### Storing Embeddings in Couchbase
Each embedding needs to be stored as a couchbase document. According to provided search index, embedding vector values need to be stored in the `vector` field. The original text of the embedding can be stored in the same document:

In [11]:
for i in range(0, len(documents)):
    doc = {
        "id": str(uuid.uuid4()),
        "text": documents[i],
        "embedding": embeddings[i].values,
    }
    collection.upsert(doc["id"], doc)

# 3. Query the Index
Stored in Couchbase embeddings later can be searched using the vector index to, for example, find text fragments that would be the most relevant to some user-entered prompt:

In [12]:
search_embedding = make_embed_text_fn(
    "How can AI address climate challenges?"
)

search_req = search.SearchRequest.create(search.MatchNoneQuery()).with_vector_search(
    VectorSearch.from_vector_query(
        VectorQuery(
            "embedding", search_embedding[0].values, num_candidates=3
        )
    )
)
result = scope.search(
    "gemini_index", 
    search_req, 
    SearchOptions(
        limit=3, 
        fields=["embedding", "id", "text"]
    )
)
for row in result.rows():
    print("Found answer: " + row.id + "; score: " + str(row.score))
    doc = collection.get(row.id)
    pprint("Answer text: " + doc.value["text"].strip())
    print()

Found answer: 1fed5400-10d7-4f27-9767-4777378b69e6; score: 0.8860349059104919
('Answer text: Promoting environmentally and socially responsible deployment '
 'of AI Together, we can boldly and responsibly develop more tools and '
 'products that harness the power of AI to accelerate the climate progress we '
 'need')

Found answer: ef857d50-58d5-42db-a790-d4efdb62907f; score: 0.8660513162612915
('Answer text: Managing the environmental impact of AIWhile scaling these '
 'applications of AI and finding new ways to use it to accelerate climate '
 'action is crucial, we need to build AI responsibly and manage the '
 'environmental impact associated with it As AI is at an inflection point, '
 'predicting the future growth of energy use and emissions from AI compute in '
 'our data centers is challenging  Historically, data center energy '
 'consumption has grown much more slowly than demand for computing power')

Found answer: 89b44eca-1bd6-4f64-b28c-9b7a0b67d2c5; score: 0.8471375703811646

# Conclusion

And that's it! In this tutorial, you have successfully used Gemini to create embeddings, which can be stored and queries from Couchbase for RAG applications.